# Experiment: Case 001 Iron Chelation Organ Risk Record Gate

Objective: convert the iron-overload evidence map into a de-identified,
case-specific record gate for `case-001`.

Success criteria:
- no diagnosis or treatment advice;
- no patient identifiers or raw medical-record text;
- ranked records that a hematologist can request or mark unavailable;
- explicit boundary that ferritin alone cannot close cardiac or organ risk.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Final

CURRENT_LABEL: Final = "iron_packet_missing"

case_context = {
    "case_id": "case-001",
    "transfusion_schedule": "weekly, family-reported",
    "chelation_context": "daily medicine reported; exact drug unknown",
    "privacy": "de-identified; no raw local medical records in repo",
}

case_context

## Evidence Anchors

This notebook uses the existing source registry rather than making a new
clinical claim. The key source-backed rules are:

- repeated transfusion creates iron loading that must be monitored;
- ferritin is useful as a trend but can be confounded by inflammation,
  infection, hepatitis, tissue injury, and chelator context;
- liver iron concentration and cardiac `T2*` can be discordant;
- chelator identity, adherence, adverse effects, and kidney/liver/neutrophil
  monitoring change safety interpretation;
- endocrine, bone, heart, liver, infection, and spleen context are part of the
  organ-risk packet.


In [ ]:
@dataclass(frozen=True)
class RecordGate:
    """A missing record that blocks case-specific iron-risk interpretation."""

    name: str
    requested_record: str
    immediate_risk: int
    interpretation_value: int
    referral_value: int
    safety_value: int
    missing: bool
    blocker_label: str
    evidence_anchor: str

    def score(self) -> int:
        """Rank missing records by clinical-risk and research-use pressure."""
        if not self.missing:
            return 0

        return (
            self.immediate_risk
            + self.interpretation_value
            + self.referral_value
            + self.safety_value
        )


gates = [
    RecordGate(
        name="cardiac_t2star_mri",
        requested_record=(
            "Cardiac T2* MRI date, method, center calibration, and LVEF context."
        ),
        immediate_risk=5,
        interpretation_value=5,
        referral_value=4,
        safety_value=5,
        missing=True,
        blocker_label="cardiac_t2star_known",
        evidence_anchor="TIF 2025 iron-overload chapter; cardiac T2* literature",
    ),
    RecordGate(
        name="liver_iron_concentration_mri",
        requested_record=(
            "Liver iron concentration by MRI with date, method, and units."
        ),
        immediate_risk=4,
        interpretation_value=5,
        referral_value=4,
        safety_value=4,
        missing=True,
        blocker_label="lic_known",
        evidence_anchor="TIF 2025 LIC monitoring guidance",
    ),
    RecordGate(
        name="chelator_identity_and_plan",
        requested_record=(
            "Chelator name, dose, schedule, adherence barriers, side effects, and plan."
        ),
        immediate_risk=4,
        interpretation_value=4,
        referral_value=3,
        safety_value=5,
        missing=True,
        blocker_label="chelation_review_needed",
        evidence_anchor="TIF 2025 chelation monitoring; chelation safety snapshots",
    ),
    RecordGate(
        name="ferritin_trend_context",
        requested_record=(
            "Serial ferritin values with dates and inflammation, "
            "infection, or hepatitis context."
        ),
        immediate_risk=3,
        interpretation_value=4,
        referral_value=3,
        safety_value=3,
        missing=True,
        blocker_label="ferritin_trend_only",
        evidence_anchor="TIF 2025 ferritin trend and confounding guidance",
    ),
    RecordGate(
        name="chelator_toxicity_labs",
        requested_record=(
            "Kidney, liver, blood-count, hearing, eye, GI, "
            "and interaction monitoring as relevant to the chelator."
        ),
        immediate_risk=3,
        interpretation_value=3,
        referral_value=3,
        safety_value=5,
        missing=True,
        blocker_label="toxicity_review_needed",
        evidence_anchor="Chelation safety and drug-specific monitoring literature",
    ),
    RecordGate(
        name="endocrine_bone_packet",
        requested_record=(
            "Glucose, thyroid, parathyroid, gonadal axis, vitamin D, "
            "growth/puberty history if relevant, and bone health."
        ),
        immediate_risk=3,
        interpretation_value=3,
        referral_value=3,
        safety_value=4,
        missing=True,
        blocker_label="organ_screen_incomplete",
        evidence_anchor=(
            "TIF monitoring recommendations; endocrine iron-overload literature"
        ),
    ),
    RecordGate(
        name="transfusion_iron_input",
        requested_record=(
            "Unit count, volume, body weight, product type, "
            "hematocrit or pure red-cell volume, and interval."
        ),
        immediate_risk=3,
        interpretation_value=4,
        referral_value=4,
        safety_value=2,
        missing=True,
        blocker_label="iron_packet_missing",
        evidence_anchor="TIF 2025 transfusional iron-loading estimates",
    ),
    RecordGate(
        name="ferritin_confounders",
        requested_record=(
            "Recent infection, hepatitis/liver disease, inflammation, "
            "immune flare, or tissue injury near ferritin draws."
        ),
        immediate_risk=2,
        interpretation_value=4,
        referral_value=2,
        safety_value=3,
        missing=True,
        blocker_label="organ_screen_incomplete",
        evidence_anchor="TIF 2025 ferritin-confounding guidance",
    ),
]

ranked_gates = sorted(gates, key=lambda gate: (-gate.score(), gate.name))
[(gate.name, gate.score(), gate.blocker_label) for gate in ranked_gates]

## Case Packet Output

The output intentionally asks for records, not for a therapy change. A clinician
can mark each record present, absent, or not applicable. Until then, the case
stays blocked for patient-specific candidate ranking.


In [ ]:
def current_case_label(record_gates: list[RecordGate]) -> str:
    """Return the case label while required iron-risk records are missing."""
    if any(gate.missing for gate in record_gates):
        return CURRENT_LABEL

    return "specialist_managed"


doctor_packet = {
    "case_id": case_context["case_id"],
    "current_label": current_case_label(gates),
    "top_record_requests": [gate.requested_record for gate in ranked_gates[:5]],
    "secondary_record_requests": [gate.requested_record for gate in ranked_gates[5:]],
    "active_blocker_labels": sorted(
        {gate.blocker_label for gate in gates if gate.missing}
    ),
    "boundary": (
        "Ferritin alone cannot close liver, cardiac, endocrine, chelation, "
        "or trial/referral risk for this case."
    ),
}

doctor_packet

## Interpretation

Working label: `iron_packet_missing`.

For `case-001`, weekly transfusion plus reported daily chelation makes the
iron/chelation packet a high-value blocker. The top requests are cardiac `T2*`,
liver iron concentration, chelator identity/plan, ferritin trend context, and
chelator-toxicity monitoring.

This does not say the patient should change transfusion, chelation, diet,
supplements, or medicines. It says Nakafa Lab cannot make a patient-relevance
claim until the iron input, storage, organ distribution, chelation coverage,
and toxicity context are known or explicitly unavailable.
